# Video to Inverse Dynamics Pipeline

This notebook shows the full staged path from single-camera video to inverse dynamics. It includes pose cleanup, OpenSim preflight checks, estimated external loads, optional carried loads, and a notebook visualizer.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import monomech as mm

VIDEO_PATH = Path("data/subject01.mp4")
OUTPUT_DIR = Path("outputs/subject01_full_pipeline")
BODY_MASS_KG = 75.0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Pose estimation and cleanup

In [ ]:
pose = mm.estimate_pose(VIDEO_PATH, root_centered=False, floored=True)
pose = mm.smooth(pose, cutoff_hz=6.0)
pose = mm.gap_fill(pose, max_gap_frames=12)

display(pose.summary().head(12))
pose.vis_2d(frame=50)
pose.vis_3d(frame=50)

pose_csv = pose.to_csv(OUTPUT_DIR / "pose.csv")
pose_trc = pose.to_trc(OUTPUT_DIR / "pose.trc", model_path=mm.get_builtin_osim_model("pose"))
print(pose_csv)
print(pose_trc)

## 3. Scale and IK

In [ ]:
scale = mm.run_scaling(
    pose,
    model="pose",
    output_dir=OUTPUT_DIR / "scale",
)

ik = mm.run_ik(
    scale,
    output_dir=OUTPUT_DIR / "ik",
)

display(scale.summary())
display(ik.to_dataframe().head())
ik.plot()

## 4. External loads

Estimated GRF is useful for pipeline testing. Add carried or measured loads when the task includes them.

In [ ]:
estimated_grf = mm.estimate_grf(pose, body_mass_kg=BODY_MASS_KG)

# Example carried load. Delete this if the trial has no object in the hand.
dumbbell = mm.load(type="carried", body="hand_r", mass_kg=10.0)

forces = mm.external_forces(loads=[dumbbell, *estimated_grf])
print([load.name for load in forces])

## 5. Inverse dynamics

In [ ]:
id_result = mm.run_id(
    ik=ik,
    external_forces=forces,
    output_dir=OUTPUT_DIR / "id",
)

print("ID:", id_result.path)
print("External loads MOT:", id_result.metadata.get("external_loads_mot_path"))
display(id_result.to_dataframe().head())
id_result.plot()

## 6. Visualize IK, ID, and forces

In [ ]:
viewer = mm.animate(
    ik=ik,
    id=id_result,
    external_loads_path=id_result.metadata["external_loads_mot_path"],
    output_dir=OUTPUT_DIR / "visualizer",
)
viewer.show()

## 7. One-call pipeline option

Use the one-call pipeline for repeatable runs after the staged settings are working.

In [ ]:
result = mm.video_pipeline(
    VIDEO_PATH,
    model_path=mm.get_builtin_osim_model("pose"),
    output_dir=OUTPUT_DIR / "one_call",
    body_mass_kg=BODY_MASS_KG,
)

result.display()